In [29]:
import os
import fitz  # PyMuPDF

pdf_folder = "pdf"
documents = []

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        with fitz.open(os.path.join(pdf_folder, filename)) as doc:
            text = ""
            for page in doc:
                text += page.get_text()
            documents.append(text)

print(f"{len(documents)} PDF chargés.")

104 PDF chargés.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [30]:
# Initialisation du text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""]
)

docs = []

# Création des petits morceaux (chunks)
for i, text in enumerate(documents):
    splits = text_splitter.split_text(text)
    for j, split in enumerate(splits):
        docs.append(Document(
            page_content=split,
            metadata={"source": f"doc_{i+1}", "chunk": j + 1}
        ))

print(f"{len(docs)} morceaux créés.")

19506 morceaux créés.


In [5]:
!pip install chromadb langchain sentence-transformers

In [33]:
from tqdm import tqdm
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Supposons que docs est ta liste de documents (chunks)
texts = [doc.page_content for doc in docs]

# 1. Calculer manuellement les embeddings avec tqdm
embeddings = []
for text in tqdm(texts, desc="Calcul des embeddings"):
    emb = embedding_model.embed_query(text)  # embed_query pour transformer texte en vecteur
    embeddings.append(emb)

# 2. Créer la base Chroma en passant documents + vecteurs
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

# 3. Sauvegarder la base
vectordb.persist()
print(f"Base Chroma créée et sauvegardée dans 'chroma_db'.")


Calcul des embeddings: 100%|███████| 19506/19506 [1:21:23<00:00,  3.99it/s]


Base Chroma créée et sauvegardée dans 'chroma_db'.


In [23]:
from langchain.prompts import PromptTemplate

# Créer un modèle de prompt pour intégrer le contexte et la question
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Vous êtes un expert en nutrition. Répondez clairement et professionnellement à la question suivante. "
        "Utilisez d'abord les informations fournies dans le contexte, mais n'hésitez pas à compléter avec vos connaissances si besoin.\n\n"
        "Informations :\n{context}\n\n"
        "Question : {question}\n\n"
        "Réponse :"
    )
)


In [4]:
!pip install -U langchain langchain-community langchain-chroma langchain-ollama

In [1]:
!pip install --upgrade langchain langchain-community langchain-core
!pip install --upgrade langchain-ollama
!pip install --upgrade chromadb

In [33]:
# 📦 Imports nécessaires
from langchain.chains import RetrievalQA
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma 
from langchain.embeddings import HuggingFaceEmbeddings

# 📌 Étape 1 : Chargement du modèle d'embeddings HuggingFace
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 📌 Étape 2 : Chargement de la base vectorielle Chroma
vectordb = Chroma(
    persist_directory="chroma_db",
    embedding_function=embedding_model
)



In [35]:
# 📌 Étape 3 : Création du retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# 📌 Étape 4 : Connexion au modèle local Ollama TinyLlama
llm = Ollama(
    model="tinyllama",
    base_url="http://localhost:11434"
)


In [37]:
# 📌 Étape 5 : Création de la chaîne RAG
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [39]:
# 📌 Étape 6 : Fonction pour poser une question
def poser_question(question):
    result = rag_chain({"query": question})
    print("🧠 Réponse générée :\n", result["result"])
    print("\n📚 Documents sources utilisés :")
    for doc in result["source_documents"]:
        print("-", doc.metadata)

# ✅ Exemple
poser_question("Quels sont les meilleurs aliments pour lutter contre l’anémie ?")

🧠 Réponse générée :
 The following pieces of context are relevant to answering the question at hand:

1. The repast category includes a range of food options, such as vegetables, legumes, buttered bread or salad, protein sources (eggs, pasta, rice), dairy products, fruits, and lean meats/poultry, among others. These foods are typically accompanied by bread or sandwiches, along with sauces, oils, fats, sugars, and other added ingredients such as cheese, butter, salt, pepper, cocoa powder, nut butter, etc. This is a controlled meal that includes cruciferous vegetables like broccoli, cauliflower, kale or Swiss chard, lean protein sources like fish, poultry or beans, dairy products, fruits such as berries or citrus fruits, and legumes (e.g., lentils, chickpeas). The meal also contains bread and oils to prepare the protein sources, and fats to provide energy and flavor.

📚 Documents sources utilisés :
- {'chunk': 378, 'source': 'doc_82'}
- {'chunk': 378, 'source': 'doc_82'}
- {'chunk': 131,

In [47]:
import gradio as gr
# 💬 4. Fonction pour Gradio
def chatbot(query):
    try:
        # Forcer le LLM à répondre en français
        prompt = f"Réponds en français : {query}"
        result = rag_chain({"query": prompt})
        response = result["result"]
        sources = "\n".join([f"- {doc.metadata.get('source', 'Inconnu')}" for doc in result["source_documents"]])
        return f"🧠 Réponse :\n{response}\n\n📚 Sources :\n{sources}"
    except Exception as e:
        return f"❌ Erreur : {str(e)}"


In [49]:
# 🖼️ 5. Interface Gradio
interface = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(lines=2, placeholder="Posez une question sur la nutrition..."),
    outputs=gr.Textbox(lines=10),
    title="🤖 Chatbot Nutritionniste",
    description="Posez une question sur la nutrition"
)

In [51]:
# 🚀 6. Lancer l'interface
interface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [27]:
!pip install streamlit langchain langchain-community langchainhub sentence-transformers chromadb pymupdf